<a href="https://colab.research.google.com/github/mariaaapetrovskaya/Python/blob/main/lr3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа: Парсинг сложного HTML → Таблица → Агрегация → JSON

## Цель

Научиться извлекать конкретные данные из сложного HTML-документа с разнородной структурой, преобразовывать их в таблицу, агрегировать и сохранять в JSON.

## Исходные данные

Файл `library.html` — страница библиотечного сайта. Данные разбросаны по разным секциям, есть лишний контент (новости, объявления), неединая структура тегов.

## Задания

Выполняйте последовательно. Код пишите самостоятельно.

---

In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import json
import re

In [2]:
# Создаём сложный HTML-файл страницы библиотеки
html_content = '''<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Центральная библиотека</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 0; padding: 20px; background: #f5f5f5; }
        .header { background: #2c3e50; color: white; padding: 20px; margin: -20px -20px 20px -20px; }
        .nav { background: #34495e; padding: 10px; margin-bottom: 20px; }
        .nav a { color: white; margin-right: 15px; text-decoration: none; }
        .content { display: flex; }
        .main { flex: 2; padding-right: 20px; }
        .sidebar { flex: 1; background: #ecf0f1; padding: 15px; }
        .book { background: white; padding: 15px; margin-bottom: 15px; border-left: 4px solid #3498db; }
        .book h3 { margin-top: 0; color: #2c3e50; }
        .meta { color: #7f8c8d; font-size: 14px; }
        .meta span { margin-right: 15px; }
        .news { background: white; padding: 15px; margin-bottom: 15px; border-left: 4px solid #e74c3c; }
        .news h4 { margin-top: 0; color: #c0392b; }
        .announcement { background: #fff3cd; padding: 10px; margin-bottom: 10px; border-radius: 3px; }
        .footer { background: #34495e; color: white; padding: 15px; margin-top: 20px; text-align: center; }
        .tag { display: inline-block; background: #3498db; color: white; padding: 2px 8px; border-radius: 3px; font-size: 12px; margin-right: 5px; }
        .price { color: #27ae60; font-weight: bold; }
        .old-price { color: #95a5a6; text-decoration: line-through; margin-left: 10px; }
    </style>
</head>
<body>
    <div class="header">
        <h1>Центральная городская библиотека</h1>
        <p>Фонд: 500 000 изданий | Основана в 1925 году</p>
    </div>

    <div class="nav">
        <a href="#">Главная</a>
        <a href="#">Каталог</a>
        <a href="#">Услуги</a>
        <a href="#">О нас</a>
    </div>

    <div class="content">
        <div class="main">
            <h2>Новые поступления</h2>

            <div class="book">
                <h3>Война и мир</h3>
                <p class="meta">
                    <span>Автор: Лев Толстой</span>
                    <span>Год: 1869</span>
                    <span>Страниц: 1225</span>
                </p>
                <p>Роман-эпопея, описывающий русское общество в эпоху войн с Наполеоном.</p>
                <span class="tag">Классика</span>
                <span class="tag">Роман</span>
                <p class="price">350 руб. <span class="old-price">400 руб.</span></p>
            </div>

            <div class="book">
                <h3>Преступление и наказание</h3>
                <p class="meta">
                    <span>Автор: Фёдор Достоевский</span>
                    <span>Год: 1866</span>
                    <span>Страниц: 671</span>
                </p>
                <p>Психологический роман о студенте Раскольникове.</p>
                <span class="tag">Классика</span>
                <span class="tag">Роман</span>
                <p class="price">300 руб.</p>
            </div>

            <div class="book">
                <h3>Мастер и Маргарита</h3>
                <p class="meta">
                    <span>Автор: Михаил Булгаков</span>
                    <span>Год: 1967</span>
                    <span>Страниц: 384</span>
                </p>
                <p>Мистический роман о визите дьявола в Москву.</p>
                <span class="tag">Классика</span>
                <span class="tag">Мистика</span>
                <p class="price">400 руб. <span class="old-price">450 руб.</span></p>
            </div>

            <div class="book">
                <h3>Евгений Онегин</h3>
                <p class="meta">
                    <span>Автор: Александр Пушкин</span>
                    <span>Год: 1833</span>
                    <span>Страниц: 288</span>
                </p>
                <p>Роман в стихах — энциклопедия русской жизни.</p>
                <span class="tag">Классика</span>
                <span class="tag">Поэзия</span>
                <p class="price">250 руб.</p>
            </div>

            <div class="book">
                <h3>Собачье сердце</h3>
                <p class="meta">
                    <span>Автор: Михаил Булгаков</span>
                    <span>Год: 1925</span>
                    <span>Страниц: 128</span>
                </p>
                <p>Сатирическая повесть о профессоре Преображенском.</p>
                <span class="tag">Классика</span>
                <span class="tag">Сатира</span>
                <p class="price">200 руб. <span class="old-price">250 руб.</span></p>
            </div>

            <div class="book">
                <h3>Анна Каренина</h3>
                <p class="meta">
                    <span>Автор: Лев Толстой</span>
                    <span>Год: 1877</span>
                    <span>Страниц: 864</span>
                </p>
                <p>Роман о трагической любви замужней женщины.</p>
                <span class="tag">Классика</span>
                <span class="tag">Роман</span>
                <p class="price">380 руб.</p>
            </div>

            <div class="book">
                <h3>Отцы и дети</h3>
                <p class="meta">
                    <span>Автор: Иван Тургенев</span>
                    <span>Год: 1862</span>
                    <span>Страниц: 288</span>
                </p>
                <p>Роман о конфликте поколений и нигилизме.</p>
                <span class="tag">Классика</span>
                <span class="tag">Роман</span>
                <p class="price">280 руб. <span class="old-price">320 руб.</span></p>
            </div>

            <div class="book">
                <h3>Пиковая дама</h3>
                <p class="meta">
                    <span>Автор: Александр Пушкин</span>
                    <span>Год: 1834</span>
                    <span>Страниц: 96</span>
                </p>
                <p>Повесть о тайной комбинации карт.</p>
                <span class="tag">Классика</span>
                <span class="tag">Повесть</span>
                <p class="price">180 руб.</p>
            </div>

            <div class="book">
                <h3>Двенадцать стульев</h3>
                <p class="meta">
                    <span>Автор: Илья Ильф, Евгений Петров</span>
                    <span>Год: 1928</span>
                    <span>Страниц: 384</span>
                </p>
                <p>Сатирический роман о похождениях Остапа Бендера.</p>
                <span class="tag">Классика</span>
                <span class="tag">Сатира</span>
                <p class="price">320 руб. <span class="old-price">380 руб.</span></p>
            </div>

            <div class="book">
                <h3>Тарас Бульба</h3>
                <p class="meta">
                    <span>Автор: Николай Гоголь</span>
                    <span>Год: 1835</span>
                    <span>Страниц: 256</span>
                </p>
                <p>Историческая повесть о запорожских казаках.</p>
                <span class="tag">Классика</span>
                <span class="tag">Повесть</span>
                <p class="price">220 руб.</p>
            </div>

            <h2>Новости библиотеки</h2>

            <div class="news">
                <h4>Открытие нового читального зала</h4>
                <p>С 15 октября открыт новый читальный зал на 100 мест.</p>
                <p class="meta"><span>Дата: 10.10.2026</span></p>
            </div>

            <div class="news">
                <h4>Лекция о творчестве Достоевского</h4>
                <p>20 октября в 18:00 — лекция профессора Иванова.</p>
                <p class="meta"><span>Дата: 05.10.2026</span></p>
            </div>

            <div class="news">
                <h4>Книжная ярмарка</h4>
                <p>25–27 октября — ежегодная книжная ярмарка.</p>
                <p class="meta"><span>Дата: 01.10.2026</span></p>
            </div>
        </div>

        <div class="sidebar">
            <h3>Объявления</h3>

            <div class="announcement">
                <strong>Внимание!</strong> Изменение графика работы в праздничные дни.
            </div>

            <div class="announcement">
                <strong>Ищем волонтёров</strong> для проведения детских мероприятий.
            </div>

            <div class="announcement">
                <strong>Новый абонемент</strong> — скидка 20% на все услуги.
            </div>

            <h3>Статистика</h3>
            <p>Читателей: 15 000</p>
            <p>Книговыдача за месяц: 8 500</p>
            <p>Новых поступлений: 120</p>

            <h3>Контакты</h3>
            <p>Адрес: ул. Примерная, д. 1</p>
            <p>Телефон: +7 (999) 123-45-67</p>
            <p>Email: info@library.ru</p>
        </div>
    </div>

    <div class="footer">
        <p>© 2026 Центральная городская библиотека. Все права защищены.</p>
    </div>
</body>
</html>'''

with open('library.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print('Файл library.html создан')

Файл library.html создан


### Задание 1: Парсинг HTML (2 балла)

1. Извлеките только книги (игнорируйте новости, объявления, статистику). Для каждой книги нужно извлечь: название, автора, год, количество страниц, цену. Храните результаты в виде списка словарей.
2. Выведите количество книг и данные первой книги

In [3]:
with open('library.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

books = []

for book in soup.find_all('div', class_='book'):
    title_tag = book.find('h3')
    meta = book.find('p', class_='meta')
    price_tag = book.find('p', class_='price')

    if title_tag is None or meta is None or price_tag is None:
        continue

    values = {}
    for span in meta.find_all('span'):
        text = span.get_text(' ', strip=True)
        if text.startswith('Автор:'):
            values['Автор'] = text.replace('Автор:', '', 1).strip()
        elif text.startswith('Год:'):
            values['Год'] = text.replace('Год:', '', 1).strip()
        elif text.startswith('Страниц:'):
            values['Страницы'] = text.replace('Страниц:', '', 1).strip()

    # Берём только прямой текст внутри .price, не старую цену из .old-price.
    current_price_text = ''.join(price_tag.find_all(string=True, recursive=False)).strip()
    price_match = re.search(r'\d+', current_price_text.replace(' ', ''))

    if all(key in values for key in ('Автор', 'Год', 'Страницы')) and price_match:
        books.append({
            'Название': title_tag.get_text(' ', strip=True),
            'Автор': values['Автор'],
            'Год': values['Год'],
            'Страницы': values['Страницы'],
            'Цена': price_match.group()
        })

print('Количество книг:', len(books))
print('Первая книга:', books[0])

Количество книг: 10
Первая книга: {'Название': 'Война и мир', 'Автор': 'Лев Толстой', 'Год': '1869', 'Страницы': '1225', 'Цена': '350'}


### Задание 2: Создание таблицы (2 балла)

1. Создайте DataFrame со столбцами: `Название`, `Автор`, `Год`, `Страницы`, `Цена`
2. Обработайте данные:
   - Уберите пробелы из чисел (страницы, цена)
   - Приведите год, страницы, цену к числовому типу
   - Если авторов несколько — оставьте всех в одной строке через запятую
3. Выведите первые 5 строк таблицы

In [4]:
df = pd.DataFrame(books, columns=['Название', 'Автор', 'Год', 'Страницы', 'Цена'])

for column in ['Год', 'Страницы', 'Цена']:
    df[column] = df[column].astype(str).str.replace(' ', '', regex=False).astype(int)

display(df.head())

,Название,Автор,Год,Страницы,Цена
0,Война и мир,Лев Толстой,1869,1225,350
1,Преступление и наказание,Фёдор Достоевский,1866,671,300
2,Мастер и Маргарита,Михаил Булгаков,1967,384,400
3,Евгений Онегин,Александр Пушкин,1833,288,250
4,Собачье сердце,Михаил Булгаков,1925,128,200


### Задание 3: Анализ данных (2 балла)

Выполните:

1. Посчитайте количество книг каждого автора
2. Найдите самую дорогую и самую дешёвую книгу
3. Посчитайте среднюю цену книг
4. Определите, в каком десятилетии было написано больше всего книг
5. Найдите книгу с наибольшим количеством страниц

In [5]:
author_books = df.assign(Автор=df['Автор'].str.split(', ')).explode('Автор')
print('Кол-во книг каждого автора:')
display(author_books.groupby('Автор')['Название'].count().sort_values(ascending=False).rename('Кол-во книг'))

print('Самая дорогая книга:')
display(df.loc[[df['Цена'].idxmax()], ['Название', 'Автор', 'Цена']])

print('Самая дешевая книга:')
display(df.loc[[df['Цена'].idxmin()], ['Название', 'Автор', 'Цена']])

print('Средняя цена:', round(df['Цена'].mean(), 2), 'руб.')

df['Десятилетие'] = (df['Год'] // 10) * 10
decade_counts = df.groupby('Десятилетие')['Название'].count().sort_values(ascending=False)
print('Кол-во книг по десятилетиям:')
display(decade_counts.rename('Количество книг'))
max_count = decade_counts.max()
top_decades = decade_counts[decade_counts == max_count].index.tolist()
print('Больше всего книг написано в десятилетии/десятилетиях:', ', '.join(f'{d}-е' for d in top_decades))

print('Книга с наибольшим кол-вом страниц:')
display(df.loc[[df['Страницы'].idxmax()], ['Название', 'Страницы']])

Кол-во книг каждого автора:


,Кол-во книг
Автор,
Александр Пушкин,2
Лев Толстой,2
Михаил Булгаков,2
Евгений Петров,1
Илья Ильф,1
Иван Тургенев,1
Николай Гоголь,1
Фёдор Достоевский,1


Самая дорогая книга:


,Название,Автор,Цена
2,Мастер и Маргарита,Михаил Булгаков,400


Самая дешевая книга:


,Название,Автор,Цена
7,Пиковая дама,Александр Пушкин,180


Средняя цена: 288.0 руб.
Кол-во книг по десятилетиям:


,Количество книг
Десятилетие,
1830,3
1860,3
1920,2
1870,1
1960,1


Больше всего книг написано в десятилетии/десятилетиях: 1830-е, 1860-е
Книга с наибольшим кол-вом страниц:


,Название,Страницы
0,Война и мир,1225


### Задание 4: Сохранение в JSON (1 балл)

1. Создайте словарь с результатами:
   - общее количество книг
   - статистика по авторам (количество книг, средняя цена)
   - топ-3 самых дорогих книг
   - статистика по десятилетиям
2. Сохраните в файл `library_analysis.json`
3. Прочитайте обратно и выведите содержимое

In [8]:
author_stats = author_books.groupby('Автор').agg(
    **{'Количество книг': ('Название', 'count'), 'Средняя цена': ('Цена', 'mean')}
).round(2)

decade_stats = df.groupby('Десятилетие')['Название'].count().to_dict()
top3 = df.nlargest(3, 'Цена')[['Название', 'Автор', 'Цена']].to_dict(orient='records')

analysis = {
    'Общее кол-во книг': int(len(df)),
    'Статистика по авторам': {
        author: {
            'Количество книг': int(row['Количество книг']),
            'Ср цена': float(row['Средняя цена'])
        }
        for author, row in author_stats.iterrows()
    },
    'Топ самых дорогих книг': top3,
    'Статист по десятилетиям': {str(int(decade)): int(count) for decade, count in decade_stats.items()}
}

with open('library_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis, f, ensure_ascii=False, indent=4)

with open('library_analysis.json', 'r', encoding='utf-8') as f:
    loaded_analysis = json.load(f)

print(json.dumps(loaded_analysis, ensure_ascii=False, indent=4))

{
    "Общее кол-во книг": 10,
    "Статистика по авторам": {
        "Александр Пушкин": {
            "Количество книг": 2,
            "Ср цена": 215.0
        },
        "Евгений Петров": {
            "Количество книг": 1,
            "Ср цена": 320.0
        },
        "Иван Тургенев": {
            "Количество книг": 1,
            "Ср цена": 280.0
        },
        "Илья Ильф": {
            "Количество книг": 1,
            "Ср цена": 320.0
        },
        "Лев Толстой": {
            "Количество книг": 2,
            "Ср цена": 365.0
        },
        "Михаил Булгаков": {
            "Количество книг": 2,
            "Ср цена": 300.0
        },
        "Николай Гоголь": {
            "Количество книг": 1,
            "Ср цена": 220.0
        },
        "Фёдор Достоевский": {
            "Количество книг": 1,
            "Ср цена": 300.0
        }
    },
    "Топ самых дорогих книг": [
        {
            "Название": "Мастер и Маргарита",
            "Автор": "Михаил Бу

### Задание 5: Фильтрация и экспорт (1 балл)

1. Отфильтруйте книги, изданные в XIX веке
2. Посчитайте, сколько книг каждого автора попало в этот период

In [17]:
books_19 = df[(df['Год'] >= 1801) & (df['Год'] <= 1900)].copy()
print('Книги, изданные в 19 веке:')
display(books_19)

authors_19 = books_19.assign(Автор=books_19['Автор'].str.split(', ')).explode('Автор')
print('Количество книг каждого автора в 19 веке:')
display(authors_19.groupby('Автор')['Название'].count().sort_values(ascending=False).rename('Количество книг'))

Книги, изданные в 19 веке:


,Название,Автор,Год,Страницы,Цена,Десятилетие
0,Война и мир,Лев Толстой,1869,1225,350,1860
1,Преступление и наказание,Фёдор Достоевский,1866,671,300,1860
3,Евгений Онегин,Александр Пушкин,1833,288,250,1830
5,Анна Каренина,Лев Толстой,1877,864,380,1870
6,Отцы и дети,Иван Тургенев,1862,288,280,1860
7,Пиковая дама,Александр Пушкин,1834,96,180,1830
9,Тарас Бульба,Николай Гоголь,1835,256,220,1830


Количество книг каждого автора в 19 веке:


,Количество книг
Автор,
Александр Пушкин,2
Лев Толстой,2
Иван Тургенев,1
Николай Гоголь,1
Фёдор Достоевский,1


### Задание на 9–10 баллов

Поработайте со вики-страницей https://ru.wikipedia.org/wiki/Лавлейс,_Ада про первого программиста Аду Лавлейс.

№1: Найти все `<h2>` и `<h3>`, извлечь текст заголовков. Результат — кортежи вида (тип_заголовка, залоговок).

№2: Найти все `<a href>`, извлечь текст ссылки и URL. Разделить на:
- внутренние (начинаются с /wiki/)
- внешние (начинаются с http)
- изображения (ссылки на .jpg, .png, .svg)
Результат — DataFrame с колонками Текст_ссылки, URL, Тип.

№3: Извлечение фактов из infobox
В правой колонке есть `<table class="infobox">`. Извлечь пары «ключ-значение»: имя, дата рождения, место рождения, профессия и т.д. Подсказка: soup.find('table', class_='infobox'), затем найти все `<tr>` и извлечь `<th>` и `<td>`.

№4: Извлечение категорий
Внизу страницы найти все категории статьи (`<div id="catlinks">`). Подсказка: soup.find('div', id='catlinks'), извлечь текст всех `<a>`.

In [16]:
import requests
import pandas as pd
import json
from bs4 import BeautifulSoup

url = 'https://ru.wikipedia.org/wiki/Лавлейс,_Ада'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
wiki_soup = BeautifulSoup(response.text, 'html.parser')

headings = [(tag.name, tag.get_text(' ', strip=True)) for tag in wiki_soup.find_all(['h2', 'h3'])]
print('Заголовки:')
for heading in headings:
    print(heading)


links = []
for a in wiki_soup.find_all('a', href=True):
    link_text = a.get_text(' ', strip=True)
    href = a['href']
    path = href.split('?', 1)[0].lower()
    if path.endswith(('.jpg', '.jpeg', '.png', '.svg')):
        link_type = 'изображение'
    elif href.startswith('/wiki/'):
        link_type = 'внутр'
    elif href.startswith('http'):
        link_type = 'внешняя'
    else:
        continue
    links.append({'Текст_ссылки': link_text, 'URL': href, 'Тип': link_type})

links_df = pd.DataFrame(links, columns=['Текст_ссылки', 'URL', 'Тип'])
display(links_df)

infobox = wiki_soup.find('table', class_='infobox')
facts = {}
if infobox:
    for row in infobox.find_all('tr'):
        key_tag = row.find('th')
        value_tag = row.find('td')
        if key_tag and value_tag:
            key = key_tag.get_text(' ', strip=True)
            value = value_tag.get_text(' ', strip=True)
            if key and value:
                facts[key] = value
print('Факты из инфобокса:')
print(json.dumps(facts, ensure_ascii=False, indent=4))


catlinks = wiki_soup.find('div', id='catlinks')

categories = [
    a.get_text(' ', strip=True)
    for a in catlinks.find_all('a')
] if catlinks else []

if 'Категории' in categories:
    categories.remove('Категории')

print('Категории:')
print(categories)

Заголовки:
('h2', 'Содержание')
('h2', 'Биография')
('h3', 'Смерть')
('h2', 'Память')
('h3', 'Критика')
('h2', 'Примечания')
('h2', 'Литература')
('h2', 'Ссылки')


,Текст_ссылки,URL,Тип
0,Заглавная страница,/wiki/%D0%97%D0%B0%D0%B3%D0%BB%D0%B0%D0%B2%D0%...,внутр
1,Содержание,/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%...,внутр
2,Избранные статьи,/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%...,внутр
3,Случайная статья,/wiki/%D0%A1%D0%BB%D1%83%D0%B6%D0%B5%D0%B1%D0%...,внутр
4,Текущие события,/wiki/%D0%9F%D0%BE%D1%80%D1%82%D0%B0%D0%BB:%D0...,внутр
...,...,...,...
503,Разработчики,https://developer.wikimedia.org,внешняя
504,Статистика,https://stats.wikimedia.org/#/ru.wikipedia.org,внешняя
505,Заявление о куки,https://foundation.wikimedia.org/wiki/Special:...,внешняя
506,,https://www.wikimedia.org/,внешняя


Факты из инфобокса:
{
    "Имя при рождении": "Августа Ада Байрон",
    "Дата рождения": "10 декабря 1815 ( 1815-12-10 )",
    "Место рождения": "Лондон , Англия , Британская империя",
    "Дата смерти": "27 ноября 1852 ( 1852-11-27 ) (36 лет)",
    "Место смерти": "Марилебон , Лондон , Англия , Британская империя",
    "Страна": "Британская империя",
    "Род деятельности": "Программист, поэт, ученый в области информатики, изобретатель, переводчик, писатель - инженер",
    "Научная сфера": "математика [ 1 ] и вычислительная техника",
    "Место работы": "Кембриджский университет",
    "Известна как": "«мать программирования » и первый программист в истории.",
    "Награды и премии": "Google Doodle"
}
Категории:
['Родившиеся 10 декабря', 'Родившиеся в 1815 году', 'Персоналии по алфавиту', 'Родившиеся в Лондоне', 'Умершие 27 ноября', 'Умершие в 1852 году', 'Умершие в Лондоне', 'Учёные Кембриджского университета', 'Учёные по алфавиту', 'Математики по алфавиту', 'Программисты по алфавиту'